# BSP Registry Tools

### Streamlined Yocto BSP Management for Embedded Linux Teams

> *From registry to built image — one command*

---

📦 `pip install bsp-registry-tools`  
🔗 [github.com/Advantech-EECC/bsp-registry-tools](https://github.com/Advantech-EECC/bsp-registry-tools)  
⚖️ Apache 2.0 License

## The Problem

Yocto / KAS builds force teams to juggle many moving parts:

| Pain point | Without tooling |
|---|---|
| Dozens of boards × releases | Copy-pasted KAS config files |
| Docker environments per build | Undocumented, team-specific setup |
| Features (OTA, secure-boot, …) | Scattered patches, no reuse |
| Reproducibility | "Works on my machine" |
| Discoverability | "What boards do we support?" |

**One registry file → single source of truth for the whole team.**

## What is BSP Registry Tools?

- **Python CLI + library** for managing Board Support Packages
- **YAML registry** as the single source of truth (devices, releases, features, presets)
- Wraps **KAS** (`kas` / `kas-container`) to drive reproducible Yocto builds
- Works **out of the box** — zero config needed on first run
- Targets: embedded Linux teams, BSP maintainers, CI/CD pipelines

```
bsp-registry-tools
├── CLI  (bsp build / list / shell / export / deploy / gather / registry / server …)
├── Python API  (BspManager, V2Resolver, RegistryFetcher, …)
└── HTTP server  (REST + GraphQL via FastAPI)
```

## Installation

### Core
```bash
pip install bsp-registry-tools
```

### Optional extras

| Extra | Installs |
|---|---|
| `[azure]` | Azure Blob Storage upload / download |
| `[aws]` | AWS S3 upload / download |
| `[server]` | FastAPI + uvicorn + Strawberry GraphQL |
| `[completions]` | Shell tab completions (argcomplete) |
| `[dev]` | pytest, coverage, ruff, … |

```bash
pip install "bsp-registry-tools[azure,completions]"
```

## Zero-Config Quick Start

No registry file needed — the tool auto-clones the default Advantech registry on first run.

In [ ]:
# First run: clones https://github.com/Advantech-EECC/bsp-registry.git → ~/.cache/bsp/registry
# Subsequent runs: git-pull to keep up to date
!bsp list

In [ ]:
# Skip network update (great for CI or offline use)
!bsp --no-update list

In [ ]:
# Point to a different remote / branch
!bsp --remote https://github.com/my-org/bsp-registry.git --branch dev list

## Registry Resolution Priority

The tool finds the registry in this order:

1. `--registry <path>` — explicit local file, no network access
2. `--local` — use `./bsp-registry.yaml` in CWD, no network access
3. `./bsp-registry.yaml` auto-detected in CWD
4. `./bsp-registry.yml` auto-detected in CWD
5. **Remote clone** → `~/.cache/bsp/registry` (default)

```bash
# Explicit local registry
bsp --registry /path/to/my-registry.yaml list

# Force local directory, never hit network
bsp --local list

# Use a named remote saved with `bsp remotes add`
bsp --remote my-remote list
```

## Registry Schema v1 — Simple Flat YAML

Four sections: `specification`, `environment`, `containers`, `registry.bsp`.

```yaml
specification:
  version: "1.0"

environment:
  - name: "DL_DIR"
    value: "$ENV{HOME}/yocto-cache/downloads"   # $ENV{} expansion
  - name: "SSTATE_DIR"
    value: "$ENV{HOME}/yocto-cache/sstate"

containers:
  - debian-bookworm:
      image: "my-registry/debian/kas:5.1"
      file: Dockerfile
      args:
        - name: "KAS_VERSION"
          value: "5.1"

registry:
  bsp:
    - name: poky-qemuarm64-scarthgap
      description: "Poky QEMU ARM64 Scarthgap (Yocto 5.0 LTS)"
      build:
        path: build/qemu-arm64-scarthgap
        environment:
          container: "debian-bookworm"
        configuration:
          - kas/scarthgap.yaml
          - kas/qemu/qemuarm64.yaml
```

> ✅ Great for small single-team setups. Upgrade to v2 when you have many boards × releases.

## Registry Schema v2 — Separation of Concerns

v2 decomposes the registry into independent, reusable sections:

| Section | Purpose |
|---|---|
| `devices` | Hardware board definitions (slug, vendor, SoC, KAS includes) |
| `releases` | Yocto / Isar release definitions (Scarthgap, Styhead, Isar v0.11 …) |
| `features` | Optional add-ons (OTA, secure-boot, systemd …) |
| `bsp` | Named presets = device + release + features |
| `frameworks` | Build-system framework definitions (Yocto, Isar) |
| `distro` | Distribution definitions (Poky, fsl-imx-xwayland, …) |
| `vendors` | Cross-release board-vendor KAS fragments |
| `include` | Split large registries across multiple files |
| `deploy` | Cloud deployment configuration (Azure / AWS) |

**Builds can be driven by a named preset _or_ by composing components directly:**

```bash
bsp build my-preset                         # named preset
bsp build --device qemuarm64 --release scarthgap  # component-based
```

## Registry v2 — Devices & Releases

```yaml
specification:
  version: "2.0"

registry:
  devices:
    - slug: qemuarm64
      description: "QEMU ARM64 (emulated)"
      vendor: qemu
      soc_vendor: arm
      includes:
        - kas/devices/qemu/qemuarm64.yaml

    - slug: imx8mp-adv
      description: "Advantech i.MX8M Plus board"
      vendor: advantech
      soc_vendor: nxp
      includes:
        - kas/boards/imx8mp-adv.yaml

  releases:
    - slug: scarthgap
      description: "Yocto 5.0 LTS (Scarthgap)"
      distro: poky
      includes:
        - kas/scarthgap.yaml

    - slug: styhead
      description: "Yocto 5.1 (Styhead)"
      distro: poky
      includes:
        - kas/styhead.yaml
```

## Registry v2 — Named Presets (BSP)

```yaml
registry:
  bsp:
    - name: qemuarm64-scarthgap
      description: "QEMU ARM64 — Yocto 5.0 LTS"
      device: qemuarm64
      release: scarthgap
      features: []           # no optional features

    - name: imx8mp-scarthgap-ota
      description: "i.MX8M Plus — Scarthgap with OTA"
      device: imx8mp-adv
      release: scarthgap
      vendor_release: imx-6.6.53   # vendor-specific BSP sub-release
      features:
        - ota
        - secure-boot
```

**Build the preset:**

```bash
bsp build qemuarm64-scarthgap
bsp build imx8mp-scarthgap-ota --path /mnt/fast-ssd/build
```

## Splitting Large Registries with `include`

```yaml
# registry.yaml  ← root file
specification:
  version: "2.0"

include:
  - devices/qemu.yaml       # device definitions live here
  - devices/advantech.yaml
  - releases/yocto.yaml     # release definitions live here
  - releases/isar.yaml
  - features/common.yaml

registry:
  bsp:
    - name: qemuarm64-scarthgap
      device: qemuarm64
      release: scarthgap
      features: []
```

| Merge rule | Behaviour |
|---|---|
| Lists (`devices`, `releases`, `features`) | Concatenated — included items first |
| Dicts (`containers`, `environments`) | Merged recursively — including file wins |
| Scalars | Including file wins |

> Nested includes and circular-include detection are both supported.

## Feature System

Features are optional, composable add-ons injected into any build.

```yaml
registry:
  features:
    - slug: ota
      description: "Over-the-Air Update via SWUpdate"
      compatible_with: [yocto]   # only compatible with Yocto framework
      includes:
        - kas/features/ota.yaml
      local_conf:
        - "DISTRO_FEATURES:append = ' swupdate'"

    - slug: secure-boot
      description: "Secure Boot (NXP HABv4 / AHAB)"
      compatibility:
        soc_vendor: [nxp]         # only for NXP silicon
      includes:
        - kas/features/secure-boot.yaml
      env:
        - name: "SIGNING_KEY"
          value: "$ENV{SIGNING_KEY}"

    - slug: isar-users
      description: "Non-root sample users (Isar only)"
      compatible_with: [isar]
      includes:
        - kas/isar/features/users.yaml
```

```bash
bsp build --device imx8mp-adv --release scarthgap --features ota,secure-boot
```

## Vendor Overrides — Per-Vendor KAS Fragments

Releases can carry vendor-specific additions without duplicating the release definition:

```yaml
registry:
  releases:
    - slug: scarthgap
      distro: poky
      includes:
        - kas/scarthgap.yaml
      vendor_overrides:
        - vendor: advantech
          distro: fsl-imx-xwayland  # overrides distro for Advantech boards
          includes:
            - kas/advantech/scarthgap-common.yaml
          soc_vendors:
            - vendor: nxp
              includes:
                - kas/advantech/nxp/scarthgap.yaml
              releases:
                - slug: imx-6.6.53
                  includes: [kas/advantech/nxp/imx-6.6.53.yaml]
                - slug: imx-6.12.0
                  includes: [kas/advantech/nxp/imx-6.12.0.yaml]
            - vendor: mediatek
              distro: mt-distro
              includes:
                - kas/advantech/mediatek/scarthgap.yaml
```

> `soc_vendor` on the device drives automatic SoC override selection — no explicit flag needed.

## Core CLI — List & Discover

In [ ]:
# List all named presets
!bsp list

In [ ]:
# List devices
!bsp list devices

In [ ]:
# List releases
!bsp list releases

In [ ]:
# List available optional features
!bsp list features

In [ ]:
# List container definitions
!bsp containers

## Core CLI — Build

In [ ]:
# Build a named preset
!bsp build qemuarm64-scarthgap

In [ ]:
# Build by composing components directly (no preset required)
!bsp build --device qemuarm64 --release scarthgap

In [ ]:
# Checkout / validate only — fast, no actual build
!bsp build qemuarm64-scarthgap --checkout

In [ ]:
# Override the build output directory
!bsp build qemuarm64-scarthgap --path /mnt/fast-ssd/build

In [ ]:
# Clean build dir first, then build
!bsp build qemuarm64-scarthgap --clean

## Core CLI — Shell & Export

In [ ]:
# Interactive shell inside the build container
# (opens a terminal — not suitable for notebook demo)
# !bsp shell qemuarm64-scarthgap

# Execute a single command instead
!bsp shell qemuarm64-scarthgap --command "bitbake -e core-image-minimal | grep ^MACHINE="

In [ ]:
# Export the resolved KAS configuration to stdout
!bsp export qemuarm64-scarthgap

In [ ]:
# Save the exported config to a file for archiving or sharing
!bsp export qemuarm64-scarthgap --output /tmp/qemuarm64-scarthgap.yaml
!cat /tmp/qemuarm64-scarthgap.yaml

## Remote Registries & Remotes Management

Save frequently-used registry URLs as named remotes:

In [ ]:
# Add a named remote (stored in ~/.config/bsp/remotes.yaml)
!bsp remotes add upstream https://github.com/Advantech-EECC/bsp-registry.git
!bsp remotes add my-org https://github.com/my-org/bsp-registry.git

In [ ]:
# Show all saved remotes
!bsp remotes show

In [ ]:
# Use a saved remote by name
!bsp --remote my-org list

# Or by full URL with an optional branch specifier
!bsp --remote https://github.com/my-org/bsp-registry.git --branch dev list

In [ ]:
# Rename / update a remote
!bsp remotes rename my-org production
!bsp remotes set-url production https://github.com/my-org/bsp-registry-prod.git

# Remove a remote
!bsp remotes remove upstream

## Multi-Registry Mode

Load several independent registries simultaneously — useful for combining a vendor registry with your own customisations.

```bash
# Compose multiple registries on the CLI
bsp --registry vendor.yaml --registry my-additions.yaml list

# Disambiguate with registry:preset syntax when names overlap
bsp build vendor:qemuarm64-scarthgap
bsp build my-additions:custom-preset
```

```python
from bsp import BspManager

manager = BspManager(
    config_paths=[
        ("vendor",       "/path/to/vendor-registry.yaml"),
        ("my-additions", "/path/to/my-registry.yaml"),
    ]
)
manager.initialize()
manager.list_bsps()   # shows [vendor] and [my-additions] prefixes
```

## Registry Management CLI — `bsp registry`

Full CRUD on every registry entity, backed by `RegistryWriter` (atomic saves, undo stack, git helpers).

In [ ]:
# Scaffold a new registry file in the current directory
!bsp registry init --output /tmp/new-registry.yaml
!cat /tmp/new-registry.yaml

In [ ]:
# Validate an existing registry file
!bsp --registry /tmp/new-registry.yaml registry validate

In [ ]:
# Add a new device entry
!bsp --registry /tmp/new-registry.yaml registry add device \
    --slug qemuarm64 \
    --description "QEMU ARM64 (emulated)" \
    --vendor qemu \
    --soc-vendor arm \
    --includes kas/devices/qemu/qemuarm64.yaml

In [ ]:
# Compare two registry versions (unified diff)
!bsp registry diff /tmp/registry-v1.yaml /tmp/registry-v2.yaml

## Cloud Artifact Deployment

Upload Yocto build outputs to **Azure Blob Storage** or **AWS S3** after a build.

### Registry configuration

```yaml
deploy:
  provider: azure                                  # or "aws"
  account_url: $ENV{AZURE_STORAGE_ACCOUNT_URL}
  container: bsp-registry-artifacts
  prefix: "{vendor}/{device}/{release}/{date}"     # template placeholders
  patterns:
    - "**/*.wic.gz"
    - "**/*.tar.bz2"
    - "**/bzImage"
  artifact_dirs:
    - build/tmp/deploy/images
    - build/tmp/deploy/sdk
  include_manifest: true     # upload SHA-256 manifest alongside artifacts
```

In [ ]:
# Deploy artifacts after an existing build
!bsp deploy qemuarm64-scarthgap

# Or build + deploy in one step
!bsp build qemuarm64-scarthgap --deploy

# Dry-run: print what would be uploaded without uploading anything
!bsp deploy qemuarm64-scarthgap --dry-run

## Artifact Gathering (Download)

Mirror of `bsp deploy` — download previously uploaded artifacts from cloud storage.

In [ ]:
# Download artifacts for a named preset
!bsp gather qemuarm64-scarthgap --output /tmp/artifacts/

# Gather by components (no preset required)
!bsp gather --device qemuarm64 --release scarthgap --output /tmp/artifacts/

### Python API

```python
from bsp import BspManager

manager = BspManager("bsp-registry.yaml")
manager.initialize()

result = manager.gather_bsp(
    preset_name="qemuarm64-scarthgap",
    output_dir="/tmp/artifacts",
)
for artifact in result.downloaded:
    print(artifact.name, artifact.size)
```

## HTTP Server — REST + GraphQL

```bash
pip install "bsp-registry-tools[server]"
```

In [ ]:
# Start the server (background process for demo)
import subprocess, time
srv = subprocess.Popen(["bsp", "server", "--port", "8080"])
time.sleep(2)  # wait for startup

In [ ]:
import requests

# REST — list all devices
r = requests.get("http://127.0.0.1:8080/api/v1/devices")
r.json()

In [ ]:
# GraphQL — query releases
query = """
{ releases { slug description yoctoVersion } }
"""
r = requests.post("http://127.0.0.1:8080/graphql", json={"query": query})
r.json()

In [ ]:
srv.terminate()

## Shell Tab Completions

Context-aware completions for every CLI argument:

```bash
pip install "bsp-registry-tools[completions]"

# Generate and activate shell completion
bsp completions bash >> ~/.bashrc && source ~/.bashrc
bsp completions zsh  >> ~/.zshrc  && source ~/.zshrc
bsp completions fish > ~/.config/fish/completions/bsp.fish
```

**What gets completed:**

| Completer | Completes |
|---|---|
| `PresetsCompleter` | All named BSP presets in the registry |
| `DevicesCompleter` | All device slugs |
| `ReleasesCompleter` | All release slugs |
| `FeaturesCompleter` | All feature slugs |
| `RemotesCompleter` | All saved remote names |

```bash
bsp build <TAB>           # → qemuarm64-scarthgap  imx8mp-scarthgap-ota  …
bsp build --device <TAB>  # → qemuarm64  imx8mp-adv  …
bsp --remote <TAB>        # → upstream  my-org  …
```

## Python API

In [ ]:
from bsp import BspManager, RegistryFetcher

# ── 1. Fetch / update the remote registry ──────────────────────────────────
fetcher = RegistryFetcher()
registry_path = fetcher.fetch_registry(
    repo_url="https://github.com/Advantech-EECC/bsp-registry.git",
    branch="main",
    update=True,
)
print("Registry path:", registry_path)

In [ ]:
# ── 2. Load registry & inspect contents ────────────────────────────────────
manager = BspManager(str(registry_path))
manager.initialize()

print("Devices:")
for dev in manager.model.registry.devices:
    print(f"  {dev.slug}: {dev.description} (vendor={dev.vendor})")

print("\nReleases:")
for rel in manager.model.registry.releases:
    print(f"  {rel.slug}: {rel.description}")

In [ ]:
# ── 3. Resolve a preset into a full build configuration ────────────────────
from bsp import V2Resolver

resolver = V2Resolver(manager.model)
config = resolver.resolve(
    device_slug="qemuarm64",
    release_slug="scarthgap",
    feature_slugs=["ota"],
)

print("KAS files:")
for f in config.kas_files:
    print(" ", f)

## Python API — EnvironmentManager & RegistryWriter

In [ ]:
import os
from bsp import EnvironmentManager, EnvironmentVariable

# $ENV{} expansion against the host environment
env_vars = [
    EnvironmentVariable(name="DL_DIR",    value="$ENV{HOME}/yocto-cache/downloads"),
    EnvironmentVariable(name="SSTATE_DIR", value="$ENV{HOME}/yocto-cache/sstate"),
]
env = EnvironmentManager(env_vars)
print("DL_DIR  →", env.get_value("DL_DIR"))
print("SSTATE  →", env.get_value("SSTATE_DIR"))

In [ ]:
# RegistryWriter: programmatic CRUD + validation
# (import shown for illustration — requires a writable registry file)
# from bsp.registry_writer import RegistryWriter
#
# writer = RegistryWriter("/path/to/registry.yaml")
# writer.add_device(slug="new-board", description="My Board",
#                   vendor="acme", soc_vendor="nxp")
# issues = writer.validate()
# if not issues:
#     writer.save()           # atomic write with backup
#     writer.git_stage()      # git add
#     writer.git_commit("Add new-board device")
print("RegistryWriter provides: add/edit/remove + validate + undo + diff + git helpers")

## CI/CD Integration

```yaml
# .github/workflows/build.yml
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Install bsp-registry-tools
        run: pip install "bsp-registry-tools[azure]"

      - name: Validate registry
        run: bsp --local registry validate

      - name: Build BSP
        run: |
          bsp --no-update build qemuarm64-scarthgap --checkout

      - name: Build & deploy artifacts to Azure
        env:
          AZURE_STORAGE_ACCOUNT_URL: ${{ secrets.AZURE_STORAGE_ACCOUNT_URL }}
        run: |
          bsp --no-update build qemuarm64-scarthgap --deploy
```

| CI pattern | Command |
|---|---|
| Gate on registry validity | `bsp registry validate` |
| Deterministic, no network | `bsp --no-update` |
| Build + upload in one step | `bsp build <preset> --deploy` |
| Separate deploy step | `bsp deploy <preset>` |
| Download from storage | `bsp gather <preset>` |

## Architecture

```
┌──────────────────────────────────────────────────────────┐
│  CLI  (bsp/cli.py)                                       │
│  HTTP Server  (bsp/server/ — FastAPI + Strawberry)       │
└─────────────────────┬────────────────────────────────────┘
                      │
┌─────────────────────▼────────────────────────────────────┐
│  BspManager  (bsp/bsp_manager.py)                        │
│    • Coordinates all operations                          │
│    • Multi-registry support                              │
└──┬──────────┬──────────┬─────────────┬───────────────────┘
   │          │          │             │
   ▼          ▼          ▼             ▼
V2Resolver  KasManager  ArtifactDeployer  ArtifactGatherer
(resolver)  (kas_mgr)   (deployer)         (gatherer)
   │                    │                  │
   │              Azure / AWS Storage Backends
   │              (bsp/storage/azure.py, aws.py)
   ▼
RegistryFetcher   RemotesManager   RegistryWriter
(registry_fetcher) (remotes_manager) (registry_writer)
```

**Data layer:** `dacite`-based dataclasses — `RegistryRoot`, `Device`, `Release`, `Feature`, `BspPreset`, `DeployConfig`, …

## Summary

| Feature | Benefit |
|---|---|
| 📋 YAML registry (v1 + v2) | Single source of truth for all boards, releases, features |
| 🌐 Auto remote fetch | Zero-config first run; teams share one registry repo |
| 🔧 KAS integration | Reproducible Yocto builds for every device × release combo |
| 🐳 Docker environments | Consistent build containers, no "works on my machine" |
| ✨ Feature system | Composable add-ons (OTA, secure-boot, …) with compatibility checks |
| ☁️ Cloud deploy/gather | Full artifact lifecycle — build → upload → download |
| 🖥️ HTTP server | REST + GraphQL access for dashboards and automation |
| 🔡 Tab completions | Fast daily CLI use |
| 🐍 Python API | Integrate into custom tools and CI scripts |

---

```bash
pip install bsp-registry-tools
bsp list
```

🔗 [github.com/Advantech-EECC/bsp-registry-tools](https://github.com/Advantech-EECC/bsp-registry-tools)